In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
import matplotlib.pyplot as plt
import numpy as np
import time
import copy
import os
from tqdm.notebook import tqdm
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Проверка доступности GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# Установка случайного зерна для воспроизводимости
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


Используемое устройство: cpu


In [2]:
# ПАРАМЕТР: вы можете изменить размер входных данных в соответствии с требованиями вашей модели
INPUT_SIZE = 224  # стандартный размер входа для многих предобученных моделей

# Определяем базовые преобразования для изображений
# МОДИФИКАЦИЯ: вы можете экспериментировать с дополнительными аугментациями
transform_train = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),  # Изменяем размер изображений с 32x32 до INPUT_SIZE
    transforms.RandomCrop(INPUT_SIZE, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))  # Нормализация для CIFAR-10
])

transform_test = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Функция для загрузки и подготовки данных
def load_dataset(data_dir='./data', batch_size=32, num_workers=2):
    print("Загрузка датасета CIFAR-10...")
    
    # Создаем директорию для данных, если она не существует
    os.makedirs(data_dir, exist_ok=True)
    
    # Загрузка тренировочной части датасета
    train_val_dataset = datasets.CIFAR10(
        root=data_dir,
        train=True, 
        download=True,
        transform=None  # Трансформации будут применены позже
    )
    
    # Загрузка тестовой части датасета
    test_dataset = datasets.CIFAR10(
        root=data_dir,
        train=False, 
        download=True,
        transform=transform_test
    )
    
    # Разделяем тренировочные данные на обучающую и валидационную выборки
    train_size = int(0.8 * len(train_val_dataset))
    val_size = len(train_val_dataset) - train_size
    trainset, valset = random_split(
        train_val_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)  # для воспроизводимости
    )
    
    # Применяем трансформации к тренировочному и валидационному наборам
    class TransformDataset(torch.utils.data.Dataset):
        def __init__(self, subset, transform=None):
            self.subset = subset
            self.transform = transform
            
        def __getitem__(self, idx):
            x, y = self.subset[idx]
            if self.transform:
                x = self.transform(x)
            return x, y
        
        def __len__(self):
            return len(self.subset)
    
    trainset_transformed = TransformDataset(trainset, transform_train)
    valset_transformed = TransformDataset(valset, transform_test)
    
    # Создаем загрузчики данных
    trainloader = DataLoader(
        trainset_transformed, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers
    )
    
    valloader = DataLoader(
        valset_transformed, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers
    )
    
    testloader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers
    )
    
    # Получаем список классов
    classes = ['самолет', 'автомобиль', 'птица', 'кошка', 'олень', 
              'собака', 'лягушка', 'лошадь', 'корабль', 'грузовик']
    
    # Вывод информации о датасете
    print(f"Размер обучающей выборки: {len(trainset)}")
    print(f"Размер валидационной выборки: {len(valset)}")
    print(f"Размер тестовой выборки: {len(test_dataset)}")
    print(f"Количество классов: {len(classes)}")
    print(f"Классы: {classes}")
    
    dataset_info = {
        'trainset': trainset_transformed,
        'valset': valset_transformed,
        'testset': test_dataset,
        'trainloader': trainloader,
        'valloader': valloader,
        'testloader': testloader,
        'classes': classes,
        'num_classes': len(classes)
    }
    
    return dataset_info

# ПАРАМЕТР: вы можете изменить batch_size
# Загружаем данные
dataset_info = load_dataset(batch_size=64)  # Можно увеличить batch_size для CIFAR-10


Загрузка датасета CIFAR-10...


100%|██████████| 170M/170M [01:26<00:00, 1.96MB/s]   


Размер обучающей выборки: 40000
Размер валидационной выборки: 10000
Размер тестовой выборки: 10000
Количество классов: 10
Классы: ['самолет', 'автомобиль', 'птица', 'кошка', 'олень', 'собака', 'лягушка', 'лошадь', 'корабль', 'грузовик']


In [ ]:
def imshow(img, title=None):
    # Денормализация изображения
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2023, 0.1994, 0.2010])
    
    img = img.numpy().transpose((1, 2, 0))
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    plt.imshow(img)
    if title is not None:
        plt.title(title)
    plt.axis('off')

# Получаем несколько обучающих изображений
dataiter = iter(dataset_info['trainloader'])
images, labels = next(dataiter)

# Отображаем изображения
plt.figure(figsize=(15, 8))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    # Изменяем размер изображения для визуализации (уменьшаем с 224x224 до удобного размера)
    img_display = transforms.Resize((64, 64))(images[i])
    imshow(img_display, title=f"Класс: {dataset_info['classes'][labels[i]]}")
plt.tight_layout()
plt.show()


In [ ]:
def evaluate_model(model, dataloader, classes):
    """
    Оценивает модель на заданном наборе данных.
    
    Аргументы:
        model: модель PyTorch
        dataloader: загрузчик данных
        classes: список классов
        
    Возвращает:
        accuracy: общая точность
        predictions: предсказания модели
        ground_truth: истинные метки
    """
    model.eval()
    
    predictions = []
    ground_truth = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Оценка"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            predictions.extend(preds.cpu().numpy())
            ground_truth.extend(labels.cpu().numpy())
    
    # Вычисляем общую точность
    accuracy = np.mean(np.array(predictions) == np.array(ground_truth))
    
    return accuracy, predictions, ground_truth

# Оцениваем модель на тестовом наборе
test_accuracy, test_predictions, test_ground_truth = evaluate_model(
    model, dataset_info['testloader'], dataset_info['classes']
)

print(f"Точность на тестовом наборе: {test_accuracy:.4f}")